# ML Lab · Module 2, Unit 9: Evaluating a forecast honestly

**Prof. Dr. Mehmet Songur** · ML Lab, Module 2

Data: the US unemployment rate (FRED: `UNRATE`, monthly, %). Table: `unemp` (columns: `date`, `rate`). **Have Gemini write the code**, then **audit it**.

**Rules**
1. Do not change the **Setup** and **Check** cells; just run them.
2. Generate code with Gemini in the task cells; read it before running.
3. If a check cell passes, it gives you a **completion code**; paste it into the box in ML Lab.

> Note: the value for October 2025 is missing (the survey could not be collected during the US federal government shutdown). Do not fill it; audit how it is handled.

In [ ]:
#@title Setup — run it, do not change it

import io, base64, urllib.request
import numpy as np
import pandas as pd

def fred_download(series_id):
    """Downloads a series from FRED; columns: date, <series_id>. FRED marks missing values with '.';
    they are deliberately turned into NaN and counted."""
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + series_id
    with urllib.request.urlopen(url) as response:
        df = pd.read_csv(io.BytesIO(response.read()))
    df.columns = ["date", series_id]
    df["date"] = pd.to_datetime(df["date"])
    df[series_id] = pd.to_numeric(df[series_id], errors="coerce")
    print(f"{series_id}: {len(df)} rows, {df[series_id].isna().sum()} missing, {df['date'].min():%Y-%m} – {df['date'].max():%Y-%m}")
    return df

unemp = fred_download("UNRATE").rename(columns={"UNRATE": "rate"}).sort_values("date").reset_index(drop=True)
_copy = unemp.copy()
CUT = pd.Timestamp("2015-01-01")

def _k(s):
    return base64.b64decode(s).decode()

def _expected():
    d = _copy.copy()
    # The calendar is kept: the lag is built before any missing month row is dropped
    d["prev"] = d["rate"].shift(1)
    valid = d["rate"].notna() & d["prev"].notna()
    trn = d[(d["date"] < CUT) & valid]
    tst = d[(d["date"] >= CUT) & valid]
    naive = float(np.sqrt(((tst["rate"] - tst["prev"]) ** 2).mean()))
    b, a = np.polyfit(trn["prev"], trn["rate"], 1)
    model = float(np.sqrt(((tst["rate"] - (a + b * tst["prev"])) ** 2).mean()))
    alld = d[valid]
    b_all, a_all = np.polyfit(alld["prev"], alld["rate"], 1)
    out = d[(d["date"] >= CUT) & ~valid]
    return dict(naive=naive, a=a, b=b, model=model, a_all=a_all, b_all=b_all, trn=trn, tst=tst, out=out)

def _finite(x):
    try:
        return np.isfinite(float(x))
    except Exception:
        return False

def _calendar_check():
    t = _copy["date"]
    if not t.is_monotonic_increasing or t.duplicated().any():
        return "the source date sequence is not increasing and unique"
    months = pd.period_range(t.min(), t.max(), freq="M")
    if not pd.PeriodIndex(t.dt.to_period("M")).equals(months):
        return "the source calendar skips a month"
    return None

def _lag_check(table, name, expected):
    if "prev" not in table.columns:
        return f"❌ '{name}' has no 'prev' column (the rate of the previous calendar month)."
    if table["date"].duplicated().any():
        return f"❌ Some months appear more than once in '{name}'."
    s = table.set_index("date")
    ok = s["rate"].notna() & s["prev"].notna()
    got = set(pd.to_datetime(s.index[ok]))
    exp = set(pd.to_datetime(expected["date"]))
    extra = sorted(got - exp)
    if extra:
        return f"❌ '{name}' has months whose lag should not be valid: {[f'{x:%Y-%m}' for x in extra[:3]]}. If a missing month's row is dropped before the lag, the next month is tied to the wrong month; keep the calendar first, then apply shift(1)."
    missing = sorted(exp - got)
    if missing:
        return f"❌ '{name}' lacks {len(missing)} computable months or their lag is empty (first {missing[0]:%Y-%m})."
    v = s.loc[expected["date"], "prev"].to_numpy(dtype=float)
    diff = np.abs(v - expected["prev"].to_numpy(dtype=float)) > 1e-9
    if diff.any():
        m = pd.Timestamp(expected["date"].iloc[int(np.argmax(diff))])
        return f"❌ In '{name}', 'prev' for {m:%Y-%m} is not the rate of the previous calendar month. The lag is tied to the wrong month."
    return None

def check_A(train, test, rmse_naive):
    for name, t in [("train", train), ("test", test)]:
        if "date" not in t.columns or "rate" not in t.columns:
            print(f"❌ '{name}' should have 'date' and 'rate' columns."); return
    problem = _calendar_check()
    if problem:
        print("❌ Unexpected source state: " + problem); return
    if pd.Timestamp(train["date"].max()) >= CUT or pd.Timestamp(test["date"].min()) < CUT:
        print("❌ The split is not by time: training should be before 2015 and test 2015 onwards. The data may have been shuffled."); return
    e = _expected()
    for name, t, exp in [("train", train, e["trn"]), ("test", test, e["tst"])]:
        problem = _lag_check(t, name, exp)
        if problem:
            print(problem); return
    if not _finite(rmse_naive):
        print("❌ 'rmse_naive' is not a finite number (NaN or infinite)."); return
    if abs(float(rmse_naive) - e["naive"]) > 1e-4:
        print("❌ The naive forecast's RMSE differs from the expected value. Is the forecast 'prev'? Were the errors squared, averaged and square-rooted?"); return
    out = [f"{x:%Y-%m}" for x in e["out"]["date"]]
    print(f"Test: {len(e['tst'])} months evaluated. Months that cannot be computed: {out} (actual or previous month's value missing).")
    print(f"✅ All checks passed (naive RMSE ≈ {e['naive']:.4f}). Your completion code: " + _k("TUwyLVNBRi04UlRR"))

def check_B(a, b, rmse_model):
    for name, x in [("a", a), ("b", b), ("rmse_model", rmse_model)]:
        if not _finite(x):
            print(f"❌ '{name}' is not a finite number (NaN or infinite). Missing values may have entered the estimation."); return
    e = _expected()
    train, test = globals().get("train"), globals().get("test")
    if abs(float(b) - e["b_all"]) < 0.002 and abs(float(b) - e["b"]) > 0.002:
        print("❌ The coefficients seem to have been estimated on all the data (including the test period): that is leakage. Use the training data only."); return
    if abs(float(b) - e["b"]) > 1e-6 or abs(float(a) - e["a"]) > 1e-6:
        print("❌ The coefficients differ from the expected ones. Is the model 'rate_t = a + b · rate_t-1'? Was it estimated on the valid training months only?"); return
    if test is None or "forecast" not in test.columns:
        print("❌ 'test' has no 'forecast' column (a + b · prev)."); return
    s = test.set_index("date")
    f = s.loc[e["tst"]["date"], "forecast"].to_numpy(dtype=float)
    if not np.allclose(f, float(a) + float(b) * e["tst"]["prev"].to_numpy(dtype=float), atol=1e-9):
        print("❌ In some months 'forecast' is not a + b · prev. Rows may be shifted or the wrong coefficients used."); return
    if abs(float(rmse_model) - e["model"]) > 1e-4:
        print("❌ The model's test RMSE differs from the expected value. Was it computed over the valid test months?"); return
    print(f"Training: {len(e['trn'])} months ({e['trn']['date'].min():%Y-%m} – {e['trn']['date'].max():%Y-%m}); test: {len(e['tst'])} months.")
    print(f"Model ≈ {e['model']:.4f}, naive ≈ {e['naive']:.4f}; difference ≈ {e['naive'] - e['model']:.4f} percentage points. One-month-ahead conditional evaluation: each month assumes the previous month's actual is known.")
    print("✅ All checks passed. Your completion code: " + _k("TUwyLU1ETC01SFdD"))

print("unemp table:", unemp.shape)
unemp.tail()

## Task A — Split by time and a benchmark

1. First **verify the calendar**: are the dates increasing and unique, with no month skipped? The October 2025 row stays with an empty value; **do not drop it**.
2. Build the lag on the full calendar: `unemp["prev"] = unemp["rate"].shift(1)`. If you build the lag after dropping missing rows, the “previous month” of November 2025 becomes September.
3. Split the table **by time**: `date < 2015-01-01` → `train`, `date >= 2015-01-01` → `test`. No shuffling. Keep the `prev` column in both tables.
4. Over the test months where both `rate` and `prev` exist, compute the RMSE of the **naive forecast** (this month = previous month) and store it in `rmse_naive`. Print the months that cannot be computed and the number of months evaluated.

**Forecast time:** each month's forecast is assumed to be made **after** the previous month's figure is published (a one-month-ahead conditional evaluation). In reality first releases can be revised later; FRED's current series contains revised values.

**What does Check A test?** That the split is by time; that every valid month's `prev` in both tables is the rate of the previous calendar month (row by row); that a month that should not be valid (November 2025) does not enter the calculation; that the RMSE is finite and correct.

**Audit questions:** Is there a `train_test_split` or `sample`? Is the cut-off date right? Was the lag built before any missing row was dropped?

In [ ]:
# Task A: generate the code with Gemini here, read it, then run it.

### Check A — run it, do not change it

In [ ]:
check_A(train, test, rmse_naive)

## Task B — A simple regression

Estimate the model `rate_t = a + b · rate_t-1` **using only the valid months of `train`** (`rate` and `prev` both present). Store the coefficients in `a` and `b`. Add a `forecast = a + b · prev` column to the test table; store the RMSE over the test period in `rmse_model` and compare it with the naive forecast.

**Scope of the comparison:** this is a one-month-ahead conditional evaluation: in each test month the previous month's actual is taken as known. Forecasting the whole test period in one go at the start of 2015 is a different (and much harder) question. If the difference is small, report it together with the test period and the data vintage (download date).

**What does Check B test?** That the coefficients are finite and estimated on the valid training months only; that `forecast` equals a + b · prev in every valid test month; that the RMSE is correct.

**Audit questions:** Which data were the coefficients estimated on (is there a `pd.concat`)? Where does last month's value for the first test month come from? By how much does the model beat the naive forecast?

In [ ]:
# Task B: generate the code with Gemini here, read it, then run it.

### Check B — run it, do not change it

In [ ]:
check_B(a, b, rmse_model)

## Your audit notes

- How did Gemini split the data? Did you need to change it?
- Which data were the regression coefficients estimated on?
- Does the model beat the naive forecast, and by how much?